# Fast co-optimization of grouped incentives (paper run)

Scalarized coordinate descent over **group incentives** (and a few **NEAT proportion MAE** ZIP-assignment warm-up passes), using a **coarse incentive grid** so a few `n_groups` values usually finish in **well under an hour**.

See `Examples/co_optimize_group_incentives.py` for the API. **Tune** `INCENTIVE_GRID` (finer = slower), `max_iters`, and `alpha_prop` (larger = weight matching NEAT proportions more).

In [ ]:
from pathlib import Path
import pickle
import random
import sys
import time

import numpy as np
import pandas as pd
from tqdm import tqdm

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "Models").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Data.data_load_util import make_dataset
from Simulation.projections_util import create_paper_objectives

sys.path.insert(0, str(repo_root / "Examples"))
from co_optimize_group_incentives import co_optimize_group_incentives, pareto_weight_sketches

In [ ]:
# --- Config: coarse grid for speed (paper deadline) ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

TARGET_ADOPTION_MODE = "additional_only"
YEARS_TO_SIMULATE = 5
REFERENCE_YEAR_INDEX = -1
REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION = 0

FEDERAL_CACHE_DIR = repo_root / "Examples" / "model_cache" / "federal"
NEAT_PROJ_PATH = repo_root / "Examples" / "Proportional_NEAT" / "PROJ_softmax_b0p15_large.pkl"

# Coarser than greedy notebook (500): fewer columns in work_df, faster coordinate search.
INCENTIVE_GRID = list(range(-2000, 8001, 1000))

N_GROUPS_TO_RUN = [3, 5, 8]
CO_OPT_ITERS = 6
ASSIGNMENT_PASSES = 2
ALPHA_PROP = 0.06

if not FEDERAL_CACHE_DIR.exists():
    raise RuntimeError("Federal cache missing.")
if not NEAT_PROJ_PATH.exists():
    raise RuntimeError(f"Missing {NEAT_PROJ_PATH}")

In [ ]:
state_behavior_df = pd.read_csv(repo_root / "Models" / "Incentives" / "state_behavior.csv")
zips_df, _, _ = make_dataset(granularity="both", remove_outliers=False, load_dir_prefix=str(repo_root / "Data") + "/")
zip_lookup = zips_df.set_index("region_name")
objectives = create_paper_objectives()

with open(NEAT_PROJ_PATH, "rb") as f:
    neat_proj = pickle.load(f)

target_panels = {}
for k, v in neat_proj.panel_placements.items():
    try:
        z = int(k)
    except Exception:
        continue
    if z in zip_lookup.index:
        target_panels[z] = float(v)

target_total = sum(target_panels.values())
target_prop = {z: (p / target_total if target_total > 0 else 0.0) for z, p in target_panels.items()}

def build_target_adoption_curve(base_adoption, annual_increment, years_total, mode):
    out = []
    for year in range(1, years_total + 1):
        if mode == "absolute_total":
            val = base_adoption + year * annual_increment
        else:
            val = year * annual_increment
        out.append(float(np.clip(val, 0.0, 1.0)))
    return out

def required_incentives(payload, cutoff):
    return payload["installation_cost"] - payload["yearly_savings_coeff"] * cutoff

def eval_cutoff(payloads, cutoff, threshold):
    vals = []
    for p in payloads:
        req = required_incentives(p, cutoff)
        vals.append(float(np.mean(req <= threshold)))
    return float(np.mean(vals)) if vals else 0.0

def calibrate_cutoffs(payloads, targets, threshold, lower=0.25, upper=30.0, steps=28):
    cutoffs = []
    prev = lower
    for t in targets:
        lo, hi = prev, upper
        if eval_cutoff(payloads, hi, threshold) < t:
            cutoffs.append(hi)
            prev = hi
            continue
        if eval_cutoff(payloads, lo, threshold) >= t:
            cutoffs.append(lo)
            prev = lo
            continue
        for _ in range(steps):
            mid = 0.5 * (lo + hi)
            if eval_cutoff(payloads, mid, threshold) < t:
                lo = mid
            else:
                hi = mid
        cutoffs.append(hi)
        prev = hi
    return cutoffs

def load_state_payloads(cache_dir):
    state_payloads = {}
    for p in sorted(cache_dir.glob("zip_payloads_*.pkl")):
        parts = p.stem.split("_")
        if len(parts) < 3:
            continue
        st = parts[2]
        with open(p, "rb") as f:
            cached = pickle.load(f)
        payloads = cached.get("payloads", [])
        if payloads:
            if st not in state_payloads or len(payloads) > len(state_payloads[st]):
                state_payloads[st] = payloads
    return state_payloads

state_payloads = load_state_payloads(FEDERAL_CACHE_DIR)
state_cutoffs = {}
for st in tqdm(sorted(state_payloads.keys()), desc="Calibrate state cutoffs", unit="state"):
    row = state_behavior_df[state_behavior_df["State code"] == st]
    if row.empty:
        continue
    row = row.iloc[0]
    curve = build_target_adoption_curve(
        float(row["prop_adopted_status_quo"][1:-1]),
        float(row["prop_adopted_per_year_average"]),
        YEARS_TO_SIMULATE,
        TARGET_ADOPTION_MODE,
    )
    th = -REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION
    state_cutoffs[st] = calibrate_cutoffs(state_payloads[st], curve, th)

rows = []
for st, payloads in state_payloads.items():
    if st not in state_cutoffs:
        continue
    cutoff = state_cutoffs[st][REFERENCE_YEAR_INDEX]
    for p in payloads:
        z = int(p["zip"])
        if z not in zip_lookup.index:
            continue
        qualified = float(zip_lookup.loc[z, "count_qualified"])
        req = required_incentives(p, cutoff)
        pred_panels_by_incentive = {}
        for inc in INCENTIVE_GRID:
            adopt = float(np.mean(req <= inc))
            pred_panels_by_incentive[inc] = max(0.0, adopt * qualified)
        rows.append(
            {
                "zip": z,
                "state_code": st,
                "count_qualified": qualified,
                "target_prop": float(target_prop.get(z, 0.0)),
                "pred_panels_by_incentive": pred_panels_by_incentive,
            }
        )

work_df = pd.DataFrame(rows)
print(f"Working ZIP count: {len(work_df)}, incentive grid size: {len(INCENTIVE_GRID)}")

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
equal_w = {
    "Carbon Offset": 0.25,
    "Energy Potential": 0.25,
    "Racial Equity": 0.25,
    "Income Equity": 0.25,
}

out_rows = []
t0_all = time.time()
for n in N_GROUPS_TO_RUN:
    t0 = time.time()
    gids, ginc, ev = co_optimize_group_incentives(
        work_df,
        n,
        zips_df,
        objectives,
        INCENTIVE_GRID,
        weights=equal_w,
        alpha_prop=ALPHA_PROP,
        max_iters=CO_OPT_ITERS,
        assignment_passes=ASSIGNMENT_PASSES,
        rng=rng,
    )
    elapsed = time.time() - t0
    row = {
        "n_groups": n,
        "elapsed_sec": elapsed,
        "group_incentives": str(ginc),
        "match_mae": ev["mae"],
        "match_score": ev["match_score"],
    }
    row.update(ev["objectives"])
    out_rows.append(row)
    print(f"n={n} done in {elapsed:.1f}s | MAE={ev['mae']:.6f} | incentives={ginc}")
    print(ev["objectives"])

print(f"Total wall time: {time.time() - t0_all:.1f}s")
summary = pd.DataFrame(out_rows)
display(summary)

In [ ]:
# Optional: a few Pareto-style sketches (different weight vectors). Increase runtime.
RUN_SKETCHES = False

if RUN_SKETCHES:
    sketches = [
        {"Carbon Offset": 0.5, "Energy Potential": 0.5, "Racial Equity": 0.0, "Income Equity": 0.0},
        {"Carbon Offset": 0.2, "Energy Potential": 0.2, "Racial Equity": 0.3, "Income Equity": 0.3},
        {"Carbon Offset": 0.1, "Energy Potential": 0.1, "Racial Equity": 0.4, "Income Equity": 0.4},
    ]
    sk_df = pareto_weight_sketches(
        work_df,
        5,
        zips_df,
        objectives,
        INCENTIVE_GRID,
        sketches,
        alpha_prop=ALPHA_PROP,
        max_iters=CO_OPT_ITERS,
        assignment_passes=ASSIGNMENT_PASSES,
    )
    display(sk_df)